# Обучение RandomForest модели для Smart AI Brain

Эта модель используется как ML Gatekeeper в Smart AI Brain для экономии API запросов.

In [ ]:
# Установка зависимостей
!pip install -q scikit-learn pandas numpy joblib ta

In [ ]:
# Загрузка данных
from google.colab import files
print("Загрузите все CSV файлы из ml_data/")
uploaded = files.upload()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import ta

print("✅ Libraries loaded")

In [ ]:
def load_and_prepare_data():
    """Загрузить и подготовить данные"""
    print("\n📂 Loading data...")
    
    all_data = []
    
    for filename in os.listdir('.'):
        if filename.endswith('.csv') and filename.startswith('klines_'):
            symbol = filename.split('_')[1]
            print(f"   Loading {filename}...")
            
            df = pd.read_csv(filename)
            df['symbol'] = symbol
            all_data.append(df)
            print(f"      {len(df)} records")
    
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"\n✅ Loaded {len(combined_df)} total records")
    
    return combined_df

df = load_and_prepare_data()
df.head()

In [ ]:
def add_technical_indicators(df):
    """Добавить технические индикаторы"""
    print("\n📊 Adding technical indicators...")
    
    df = df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    
    result_dfs = []
    
    for symbol in df['symbol'].unique():
        symbol_df = df[df['symbol'] == symbol].copy()
        
        # RSI
        symbol_df['rsi'] = ta.momentum.RSIIndicator(symbol_df['close'], window=14).rsi()
        
        # MACD
        macd = ta.trend.MACD(symbol_df['close'])
        symbol_df['macd'] = macd.macd()
        symbol_df['macd_signal'] = macd.macd_signal()
        
        # Bollinger Bands
        bb = ta.volatility.BollingerBands(symbol_df['close'])
        symbol_df['bb_upper'] = bb.bollinger_hband()
        symbol_df['bb_middle'] = bb.bollinger_mavg()
        symbol_df['bb_lower'] = bb.bollinger_lband()
        
        # EMA
        symbol_df['ema_20'] = ta.trend.EMAIndicator(symbol_df['close'], window=20).ema_indicator()
        symbol_df['ema_50'] = ta.trend.EMAIndicator(symbol_df['close'], window=50).ema_indicator()
        
        # Trend score
        symbol_df['trend_score'] = np.where(symbol_df['ema_20'] > symbol_df['ema_50'], 1, -1)
        
        # Target: price change in next period (1 hour)
        symbol_df['target'] = (symbol_df['close'].shift(-1) - symbol_df['close']) / symbol_df['close']
        
        result_dfs.append(symbol_df)
    
    result_df = pd.concat(result_dfs, ignore_index=True)
    
    # Удаляем NaN
    result_df = result_df.dropna()
    
    print(f"✅ Added indicators, {len(result_df)} records after cleaning")
    
    return result_df

df = add_technical_indicators(df)
df.head()

In [ ]:
# Подготовка данных для обучения
feature_columns = ['close', 'rsi', 'macd', 'macd_signal', 'bb_upper', 'bb_middle', 'bb_lower', 
                   'ema_20', 'ema_50', 'volume', 'trend_score']

X = df[feature_columns].values
y = df['target'].values

print(f"\n📊 Dataset shape:")
print(f"   X: {X.shape}")
print(f"   y: {y.shape}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print(f"\n✅ Train/Test split:")
print(f"   Train: {X_train.shape[0]} samples")
print(f"   Test: {X_test.shape[0]} samples")

In [ ]:
# Обучение RandomForest
print("\n🌲 Training RandomForest...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model.fit(X_train, y_train)

print("\n✅ Training complete!")

In [ ]:
# Оценка модели
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("\n📊 Model Performance:")
print("\nTrain:")
print(f"   MSE: {mean_squared_error(y_train, y_pred_train):.6f}")
print(f"   MAE: {mean_absolute_error(y_train, y_pred_train):.6f}")
print(f"   R²: {r2_score(y_train, y_pred_train):.4f}")

print("\nTest:")
print(f"   MSE: {mean_squared_error(y_test, y_pred_test):.6f}")
print(f"   MAE: {mean_absolute_error(y_test, y_pred_test):.6f}")
print(f"   R²: {r2_score(y_test, y_pred_test):.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 Feature Importance:")
print(feature_importance)

In [ ]:
# Сохранение модели
print("\n💾 Saving model...")

joblib.dump(model, 'price_predictor.pkl')

print("✅ Model saved as price_predictor.pkl")
print("\n📥 Download the file and upload to server:")
print("   scp price_predictor.pkl root@88.210.10.145:/root/Bybit_Trader/ml_data/models/")

# Download
files.download('price_predictor.pkl')

In [ ]:
# Тест предсказания
print("\n🧪 Testing predictions...")

test_samples = X_test[:10]
predictions = model.predict(test_samples)

for i, (pred, actual) in enumerate(zip(predictions, y_test[:10])):
    direction_pred = 'UP' if pred > 0.015 else ('DOWN' if pred < -0.015 else 'HOLD')
    direction_actual = 'UP' if actual > 0 else 'DOWN'
    
    print(f"{i+1}. Predicted: {pred:+.4f} ({direction_pred}) | Actual: {actual:+.4f} ({direction_actual})")